## 파서 확인 

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day05" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w5" / "day05"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))


SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
MODELS = ROOT.parent / "models"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [2]:
PDF = SAMPLES / "DOC-HR-014_국내출장_여비_규정_v2.0.pdf"
DOCX = SAMPLES / "DOC-HR-014_국내출장_여비_규정_v2.0.docx"
SCANNED = SAMPLES / "_formats" / "sample_scanned.pdf"

In [3]:
import importlib
import app.rag.local_parsers as lp
from app.rag.chunker import chunk, summarize
from app.rag.parser import parse


/Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- MD 파일

In [4]:
MD = SAMPLES / "_formats" / "sample_guide.md"

doc = parse(MD)
print(f"블록 {len(doc.blocks)} -> {summarize(chunk(doc))}")

for b in doc.blocks:
    print(f" [{b.locator}] {b.text}")

블록 3 -> 조항 단위 1 + 표 단위 0 = 1 청크
 [1. 신청] 출장 3일 전까지 그룹웨어에서 신청합니다.
 [2. 승인] 팀장 승인 후 확정됩니다. 5일 초과 출장은 본부장 승인이 필요합니다.
 [3. 정산] 복귀 후 7일 이내에 영수증을 첨부해 정산합니다.


- HTML 파일

In [5]:
HTML = FORMATS / "sample_page.html"
raw = HTML.read_text(encoding="utf-8")
title = raw.split("<title>")[1].split("</title>")[0]
body = parse(HTML).blocks[0].text
print(title)
print(body[:20])
print(body.startswith(title))

사내 공지
추석 연휴 근무 안내 연휴 기간 중 
False


* TXT 파일

In [6]:
CP949 = FORMATS / "memo_cp949.txt"

# 옛날 버전 -> 깨짐 
old = CP949.read_bytes().decode("utf-8", errors="replace")
print(old)

# CP949 파싱 가능 
print([b.text for b in parse(CP949).blocks])

���� ���� �ȳ�

9������ ���� ������ �׷������� �����մϴ�.
���� ������ ���� �ʽ��ϴ�.

����: �λ��ѹ��� ������ ����

09:11:23 INFO     app.rag.parser : 인코딩을 cp949 로 읽었습니다: memo_cp949.txt
['출장 정산 안내', '9월부터 출장 정산은 그룹웨어에서만 접수합니다.\n종이 서류는 받지 않습니다.', '문의: 인사총무팀 김지원 과장']


* XSLX 파일 

In [7]:
from openpyxl import Workbook

SANDBOX.mkdir(parents=True, exist_ok=True)
wb = Workbook()
ws = wb.active
ws.title = "출장비 집계"
ws.append(["구분", "1분기", "2분기"])
ws.append(["인프라사업부", 4200000, 3800000])
ws.append([None, 1200000, 1400000])                  
ws.append(["구매팀", 900000, 1000000])
ws.merge_cells("A2:A3")                             
MERGED = SANDBOX / "merged.xlsx"
wb.save(MERGED)

doc = lp.parse_xlsx(MERGED)
print(f"블록 {len(doc.blocks)} · 표 {doc.table_count}")
print(doc.blocks[0].text)

블록 1 · 표 1
| 구분 | 1분기 | 2분기 |
|---|---|---|
| 인프라사업부 | 4200000 | 3800000 |
| 인프라사업부 | 1200000 | 1400000 |
| 구매팀 | 900000 | 1000000 |


In [8]:
from openpyxl import Workbook
from openpyxl import load_workbook

wb = Workbook()
ws = wb.active
ws.title = "부서별"
for row in (["부서", "출장비"], ["인프라사업부", 4200000], ["구매팀", 1200000], ["합계", "=SUM(B2:B3)"]):
    ws.append(row)
FORMULA = SANDBOX / "formula.xlsx"
wb.save(FORMULA)                                    

print("data_only 로 읽은 B5 :", load_workbook(FORMULA, data_only=True).active["B5"].value)
print(lp.parse_xlsx(FORMULA).blocks[0].text)

data_only 로 읽은 B5 : None
| 부서 | 출장비 |
|---|---|
| 인프라사업부 | 4200000 |
| 구매팀 | 1200000 |
| 합계 | =SUM(B2:B3) (계산값 없음) |


In [9]:
wb = Workbook()
ws = wb.active
ws.title = "분기별"
for row in (["2026 출장 · 교육비 집계", None, None, None, None],
            ["구분", "1분기", None, "2분기", None],
            [None, "출장비", "교육비", "출장비", "교육비"],
            ["인프라사업부", 4200000, 1200000, 3800000, 1400000],
            ["구매팀", 900000, 300000, 1000000, 250000]):
    ws.append(row)
for rng in ("A1:E1", "B2:C2", "D2:E2", "A2:A3"):      
    ws.merge_cells(rng)
HEADER = SANDBOX / "header.xlsx"
wb.save(HEADER)

print(lp.parse_xlsx(HEADER).blocks[0].text)

2026 출장 · 교육비 집계

| 구분 | 1분기 출장비 | 1분기 교육비 | 2분기 출장비 | 2분기 교육비 |
|---|---|---|---|---|
| 인프라사업부 | 4200000 | 1200000 | 3800000 | 1400000 |
| 구매팀 | 900000 | 300000 | 1000000 | 250000 |


In [10]:
wb = Workbook()
ws = wb.active
ws.title = "요약"
for row in (["항목", "상한"], ["숙박", "100,000"], ["식비", "35,000"], [],
            ["직급", "일비", "숙박비", "식비"], ["1~2급", "40,000", "100,000", "35,000"]):
    ws.append(row)
MULTI = SANDBOX / "multi.xlsx"
wb.save(MULTI)

doc = lp.parse_xlsx(MULTI)
print(f"블록 {len(doc.blocks)} · 쪽 {doc.page_count} · 표 {doc.table_count}")
for b in doc.blocks:
    print(f"--- [{b.kind}] {b.locator}")
    print(b.text)

블록 2 · 쪽 1 · 표 2
--- [표] 요약 · 표1
| 항목 | 상한 |
|---|---|
| 숙박 | 100,000 |
| 식비 | 35,000 |
--- [표] 요약 · 표2
| 직급 | 일비 | 숙박비 | 식비 |
|---|---|---|---|
| 1~2급 | 40,000 | 100,000 | 35,000 |


* PPTX 파일 

In [11]:
from pptx import Presentation
from pptx.util import Inches

prs = Presentation()
slide = prs.slides.add_slide(prs.slide_layouts[6])  
slide.shapes.add_textbox(Inches(0.5), Inches(0.5), Inches(6), Inches(1)).text_frame.text = "출장비 집행 현황"
table = slide.shapes.add_table(3, 3, Inches(0.5), Inches(1.8), Inches(6), Inches(2)).table
for r, row in enumerate([["부서", "1분기", "2분기"],
                         ["인프라사업부", "4200000", "3800000"],
                         ["구매팀", "1200000", "1400000"]]):
    for c, value in enumerate(row):
        table.cell(r, c).text = value
WITH_TABLE = SANDBOX / "with_table.pptx"
prs.save(str(WITH_TABLE))

doc = lp.parse_local(WITH_TABLE)
print(f"블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}  →  {summarize(chunk(doc))}")
for b in doc.blocks:
    print(f"--- [{b.kind}] {b.locator}")
    print(b.text)

블록 2 · 표 1 · 쪽 1  →  조항 단위 1 + 표 단위 1 = 2 청크
--- [조항] 슬라이드 1
출장비 집행 현황
--- [표] 슬라이드 1 · 표1
| 부서 | 1분기 | 2분기 |
|---|---|---|
| 인프라사업부 | 4200000 | 3800000 |
| 구매팀 | 1200000 | 1400000 |


In [12]:
prs = Presentation()
slide = prs.slides.add_slide(prs.slide_layouts[6])
group = slide.shapes.add_group_shape()               
group.shapes.add_textbox(Inches(0.5), Inches(0.5), Inches(6), Inches(1)).text_frame.text = "숙박비 상한(그룹 안)"
frame = slide.shapes.add_table(2, 2, Inches(0.5), Inches(1.8), Inches(6), Inches(1))
for c, (head, value) in enumerate([("광역시", "100,000"), ("그 밖", "80,000")]):
    frame.table.cell(0, c).text = head
    frame.table.cell(1, c).text = value
group._element.append(frame._element)               
GROUPED = SANDBOX / "grouped.pptx"
prs.save(str(GROUPED))

print("맨 바깥 도형     :", [type(s).__name__ for s in Presentation(str(GROUPED)).slides[0].shapes])
doc = lp.parse_pptx(GROUPED)
print(f"블록 {len(doc.blocks)} · 표 {doc.table_count}")
for b in doc.blocks:
    print(f"--- [{b.kind}] {b.locator}")
    print(b.text)

맨 바깥 도형     : ['GroupShape']
블록 2 · 표 1
--- [조항] 슬라이드 1
숙박비 상한(그룹 안)
--- [표] 슬라이드 1 · 표1
| 광역시 | 그 밖 |
|---|---|
| 100,000 | 80,000 |


In [13]:
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE

prs = Presentation()
slide = prs.slides.add_slide(prs.slide_layouts[6])
data = CategoryChartData()
data.categories = ["1분기", "2분기"]
data.add_series("출장비", (4200000, 3800000))
data.add_series("교육비", (1200000, 1400000))
chart = slide.shapes.add_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, Inches(0.5), Inches(0.5),
                               Inches(6), Inches(4), data).chart
chart.has_title = True
chart.chart_title.text_frame.text = "분기별 집행액"
CHART = SANDBOX / "with_chart.pptx"
prs.save(str(CHART))

doc = lp.parse_pptx(CHART)
print(f"블록 {len(doc.blocks)} · 표 {doc.table_count}  →  {summarize(chunk(doc))}")
for b in doc.blocks:
    print(f"--- [{b.kind}] {b.locator}")
    print(b.text)

블록 1 · 표 1  →  조항 단위 0 + 표 단위 1 = 1 청크
--- [표] 슬라이드 1 · 차트1
분기별 집행액

| 항목 | 출장비 | 교육비 |
|---|---|---|
| 1분기 | 4200000 | 1200000 |
| 2분기 | 3800000 | 1400000 |


In [14]:
for name, fn in (("parse_pdf       ", lp.parse_pdf), ("parse_pdf_tables", lp.parse_pdf_tables)):
    doc = fn(PDF)
    print(f"{name}  블록 {len(doc.blocks):>2} · 쪽 {doc.page_count} · 표 {doc.table_count}  →  {summarize(chunk(doc))}")
docx = lp.parse_local(DOCX)
print(f"(같은 규정 .docx)  블록 {len(docx.blocks)} · 쪽 {docx.page_count} · 표 {docx.table_count}  →  {summarize(chunk(docx))}")

parse_pdf         블록 16 · 쪽 16 · 표 0  →  조항 단위 45 + 표 단위 0 = 45 청크
parse_pdf_tables  블록 23 · 쪽 16 · 표 7  →  조항 단위 46 + 표 단위 7 = 53 청크
(같은 규정 .docx)  블록 233 · 쪽 1 · 표 7  →  조항 단위 46 + 표 단위 7 = 53 청크


In [15]:
doc = lp.parse_pdf_tables(PDF)
table = next(b for b in doc.blocks if b.kind == "표" and b.text.startswith("| 직급 | 일비"))
body = "\n".join(b.text for b in doc.blocks if b.kind == "조항")

print(table.locator)
print(table.text)
print()
print("「100,000」 이 표에    :", "100,000" in table.text)
print("「100,000」 이 본문에도 :", "100,000" in body)

p.11 · 표1
| 직급 | 일비 | 숙박비(광역시) | 숙박비(그 밖의 지역) | 식비 |
|---|---|---|---|---|
| 1~2급 | 40,000 | 100,000 | 80,000 | 30,000 |
| 3급 | 35,000 | 80,000 | 70,000 | 27,000 |
| 4급 | 30,000 | 70,000 | 60,000 | 25,000 |
| 5급 이하 | 25,000 | 60,000 | 50,000 | 25,000 |

「100,000」 이 표에    : True
「100,000」 이 본문에도 : False


In [16]:
import pdfplumber
from pypdf import PdfReader
from app.core.exceptions import ValidationFailed

SCAN = SANDBOX / "DOC-HR-014_국내출장_여비_규정_v2.0_스캔본.pdf"   
reader = PdfReader(str(SCAN))
print("쪽 수             :", len(reader.pages))
print("쪽마다 든 그림     :", [len(p.images) for p in reader.pages])      
print("pypdf 글자        :", sum(len((p.extract_text() or "").strip()) for p in reader.pages))
with pdfplumber.open(str(SCAN)) as pdf:
    print("pdfplumber 글자   :", sum(len((p.extract_text() or "").strip()) for p in pdf.pages),
          "· 표", sum(len(p.find_tables()) for p in pdf.pages))
try:
    lp.parse_local(SCAN)
except ValidationFailed as e:
    print("parse_local       막힘 —", e)

쪽 수             : 3
쪽마다 든 그림     : [1, 1, 1]
pypdf 글자        : 0
pdfplumber 글자   : 0 · 표 0
09:11:24 INFO     app.rag.local_parsers : 텍스트 없음 (스캔본으로 보임): DOC-HR-014_국내출장_여비_규정_v2.0_스캔본.pdf p.1
09:11:24 INFO     app.rag.local_parsers : 텍스트 없음 (스캔본으로 보임): DOC-HR-014_국내출장_여비_규정_v2.0_스캔본.pdf p.2
09:11:24 INFO     app.rag.local_parsers : 텍스트 없음 (스캔본으로 보임): DOC-HR-014_국내출장_여비_규정_v2.0_스캔본.pdf p.3
parse_local       막힘 — 문서에서 글자를 찾지 못했습니다: DOC-HR-014_국내출장_여비_규정_v2.0_스캔본.pdf


In [17]:
from app.core.config import get_settings

settings = get_settings()
HAS_KEY = bool(settings.upstage_api_key)         

# 정답지 
TRUTH = ["40,000", "100,000", "80,000", "30,000", "35,000", "70,000", "27,000", "60,000", "25,000", "50,000"]

if HAS_KEY:
    from app.integrations.upstage import UpstageParser

    before = settings.upstage_parse_ocr
    settings.upstage_parse_ocr = "force"         
    try:
        ocr_doc = UpstageParser().parse(str(SCAN))  
    finally:
        settings.upstage_parse_ocr = before      
    text = "\n".join(b.text for b in ocr_doc.blocks)
    print(f"OCR  블록 {len(ocr_doc.blocks)} · 쪽 {ocr_doc.page_count} · 표 {ocr_doc.table_count} · 글자 {len(text)}")
    print("별표1 값 중 OCR 에 나온 것 :", sum(v in text for v in TRUTH), "/", len(TRUTH))
    print("설정은 원래대로 : ocr =", settings.upstage_parse_ocr)
else:
    print("키가 없습니다 — 이 셀은 부르지 않고 지나갑니다(비용 0)")

09:11:34 INFO     app.integrations.upstage : 상용 파싱 완료: DOC-HR-014_국내출장_여비_규정_v2.0_스캔본.pdf, 블록 25, 표 1
OCR  블록 25 · 쪽 3 · 표 1 · 글자 1530
별표1 값 중 OCR 에 나온 것 : 10 / 10
설정은 원래대로 : ocr = auto


In [18]:
if not SAMPLES.exists():
    print('재료 폴더가 없습니다 :', SAMPLES)
    print('W5의 day01안 samples 폴더를 확인해보세요.')
else:
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]
    top_files = [p for p in SAMPLES.iterdir() if p.is_file()]
    fmt_files = [p for p in FORMATS.iterdir() if p.is_file()] if FORMATS.exists() else []
    print(f'재료 파일 : {len(all_files)}개')
    print(f'  samples 바로 아래 : {len(top_files)}개 · _formats 안 : {len(fmt_files)}개')
    print('오늘 쓰는 재료 :')
    for p in sorted(SAMPLES.glob('DOC-FI-009*')):
        print(f'  {p.name}')

재료 파일 : 32개
  samples 바로 아래 : 25개 · _formats 안 : 7개
오늘 쓰는 재료 :
  DOC-FI-009_법인카드_사용_지침_v1.4.docx
  DOC-FI-009_법인카드_사용_지침_v1.4.pdf
